In [1]:
import jax
import jax.numpy as jnp

import flax.linen as nn
from flax.training import train_state
import math
import optax
from dataclasses import dataclass
import numpy as np

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:88: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [3]:
!pip install --upgrade "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

Looking in links: https://storage.googleapis.com/jax-releases/libtpu_releases.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.8/212.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.8 MB/s eta 0:00:00
  Attempting uninstall: libtpu
    Found existing installation: libtpu 0.0.21
    Uninstalling libtpu-0.0.21:
      Successfully uninstalled libtpu-0.0.21
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
  Attempting uninstall: jax
    Found existing installation: jax 0.7.2
    Uninstalling jax-0.7.2:
      Successfully uninstalled jax-0.7.2


Data Generation Functions

In [2]:
VOCAB="0123456789+= "
MAX_SEQ_LEN=len("999+999=1998")+1

stoi={c:i for i,c, in enumerate(VOCAB)}
itos={i:c for c,i in stoi.items()}

PAD = stoi[" "]

def encode(text):
  ids=[]
  for i in text:
    ids.append(stoi[i])
  if len(ids)<MAX_SEQ_LEN:
    ids+=[PAD]*(MAX_SEQ_LEN-len(ids))
  return jnp.array(ids, dtype=jnp.int32)

def decode(arr):
  ints=[]
  for i in arr:
    #need to be changed to ints because jax will store individual values as arrays
    ints.append(itos[int(i)])
  return "".join(ints)

def create_sample(a,b):
  return encode(f"{a}+{b}={a+b}")

def create_batches(batch_size):
  a = np.random.randint(0,1000,size=(batch_size,))
  b = np.random.randint(0,1000,size=(batch_size,))
  data = jnp.array([create_sample(i,j) for i,j, in zip(a,b)])

  x = data[:,:-1]
  y = data[:,1:]

  return x,y

Transformer Implementation Functions



In [6]:
@dataclass(frozen=True)
class Config:
  d_model: int = 384
  n_heads: int = 6
  n_layers: int = 6
  vocab_size: int = len(VOCAB)
  max_seq_len: int = MAX_SEQ_LEN
  n_experts: int = 8
  use_moe: bool = False
  ff_d_ratio: int=4
  use_kernel: bool = False



class Attention(nn.Module):
  config: Config
  # use compact so we dont need to specify shapes aside from output shapes
  @nn.compact
  def __call__(self,x):
    B,T,D = x.shape
    qkv = nn.Dense(3*D, use_bias=False)(x)
    q,k,v = jnp.split(qkv, 3, axis=-1)

    head_dim = self.config.d_model//self.config.n_heads

    #reshape from BTD -> BTNH
    q = q.reshape(B,T,self.config.n_heads,head_dim)
    k = k.reshape(B,T,self.config.n_heads,head_dim)
    v = v.reshape(B,T,self.config.n_heads,head_dim)

    #swapping T <-> N so we can compute our matmuls over the sequences
    q = jnp.swapaxes(q, 1, 2)
    k = jnp.swapaxes(k, 1, 2)
    v = jnp.swapaxes(v, 1, 2)

    #Q (BNTH) dot K transposed (BNHT). Since Jax ignores the first 2 dims during matmuls, we end up with BNTT
    att = jnp.matmul(q,jnp.swapaxes(k,-1,-2))

    att = att/math.sqrt(head_dim)

    #triangle mask
    mask = jnp.tril(jnp.ones((T,T)))

    #values that are not 1 are set to neg inf, so when we softmask they essentially become 0
    att = jnp.where(mask[None,None,:,:]==1, att, -1e9)

    #softmax over keys
    att = jax.nn.softmax(att, axis=-1)

    #QK (BNTT) dot V (BNTH) -> BNTH
    out = jnp.matmul(att,v)

    #need to swap these axes (N and T)
    out = jnp.swapaxes(out,1,2)

    #Now we have BTNH, but we want BTD, so we need to combine N and H, since NH=D
    out = out.reshape(B,T,D)

    #Now we multiply by W out
    out = nn.Dense(D)(out)

    return out

class MLP(nn.Module):
  config: Config
  @nn.compact
  def __call__(self,x):
    #W in
    x = nn.Dense(4*self.config.d_model)(x)
    x = nn.gelu(x)
    #W out
    x = nn.Dense(self.config.d_model)(x)
    return x

class MOE(nn.Module):
  config: Config
  @nn.compact
  def __call__(self, x):
    E = self.config.n_experts
    ff_d = self.config.ff_d_ratio

    B,T,D = x.shape
    x_flat = x.reshape(B*T,D)
    router_logits = nn.Dense(E, use_bias=False)(x_flat) #B*T, E
    router_probs = jax.nn.softmax(router_logits, axis=-1) # softmax along axis E (experts)



    top_1_probs, top_1_indices = jax.lax.top_k(router_probs, k=1) #B*T, 1

    probs_flat = top_1_probs.reshape(-1) #B,T
    indices_flat = top_1_indices.reshape(-1) #B,T

    sort_idx = jnp.argsort(indices_flat) #no need to specify axis since indices are flattened

    #sorting everything by expert
    x_sorted = x_flat[sort_idx]
    probs_sorted = probs_flat[sort_idx]

    #In/Out weights
    W_in = self.param("W_in", nn.initializers.lecun_normal(), (E,D,ff_d*D)) #project to ff_d
    W_out = self.param("W_out", nn.initializers.lecun_normal(), (E,ff_d*D,D)) #collapse back to D

    group_size = jnp.bincount(indices_flat, length=E)

    if self.config.use_kernel:
      out = fused_ragged_moe(x_sorted, W_in, W_out, group_size, BLOCK_M=128, BLOCK_F=128)
    else:
      h = jax.lax.ragged_dot(x_sorted, W_in, group_size)
      h = nn.gelu(h)
      out = jax.lax.ragged_dot(h, W_out, group_size)

    #multiply out by the sorted probs (gate)
    out = out * probs_sorted[:, None]

    #get unsorting idx list
    unsort_idx = jnp.argsort(sort_idx) #pretty neat because the sort_idx is a list containing all original indices

    out = out[unsort_idx]

    out = out.reshape(B,T,D) #go from (B*T, D) -> (B,T,D)

    return out

class Transformer(nn.Module):
  config: Config
  @nn.compact
  def __call__(self, x):
    #residual connections and layernorms before each attention and mlp layer
    x = x + Attention(self.config)(nn.LayerNorm()(x))
    if self.config.use_moe:
      x = x + MOE(self.config)(nn.LayerNorm()(x))
    else:
      x = x + MLP(self.config)(nn.LayerNorm()(x))
    return x

class LanguageModel(nn.Module):
  config:Config
  @nn.compact
  def __call__(self, tokens):
    _,T = tokens.shape
    #VD embedding for tokens
    tok_emb = nn.Embed(self.config.vocab_size, self.config.d_model)(tokens)
    #get pos emb params and set std to 0.02, as per gpt2 paper
    pos_emb = self.param("pos_emb", nn.initializers.normal(0.02), (self.config.max_seq_len, self.config.d_model))

    #cut off pos_emb because it may be longer than the sequence
    x = tok_emb+pos_emb[:T]

    #now we can send x through transformer blocks
    for _ in range(self.config.n_layers):
      x=Transformer(self.config)(x)

    #Final layer norm
    x = nn.LayerNorm()(x)

    #shape of x right now is BTD, we need BTV
    logits = nn.Dense(self.config.vocab_size)(x)

    return logits


Pallas kernel

In [4]:
from jax.experimental import pallas as pl
from functools import partial

def fused_ragged_moe_kernel(x_ref, w_in_ref, w_out_ref, block_experts_ids, y_ref, *, BLOCK_M, BLOCK_F, D, F, MAX_BLOCKS):
  block_id = pl.program_id(0)

  all_expert_ids = block_experts_ids[pl.dslice(0, MAX_BLOCKS)]

  expert_id = jnp.sum(
      jnp.where(
          jnp.arange(MAX_BLOCKS, dtype=jnp.int32) == block_id,
          all_expert_ids,
          0
      )
  )

  token_start = block_id * BLOCK_M

  x_tile = x_ref[pl.dslice(token_start,BLOCK_M), pl.dslice(0,D)]

  acc = jnp.zeros((BLOCK_M, D), dtype=jnp.float32)

  for f0 in range(0, F, BLOCK_F):
    w_in_tile = w_in_ref[expert_id, pl.dslice(0,D), pl.dslice(f0, BLOCK_F)]
    h = jnp.dot(x_tile, w_in_tile)

    h = nn.gelu(h)

    w_out_tile = w_out_ref[expert_id, pl.dslice(f0, BLOCK_F), pl.dslice(0,D)]

    acc+=jnp.dot(h, w_out_tile)

  y_ref[pl.dslice(token_start, BLOCK_M), pl.dslice(0,D)] = acc.astype(y_ref.dtype)


def fused_ragged_moe(x_sorted, w_in, w_out, group_sizes, BLOCK_M=128, BLOCK_F=128):
  M,D = x_sorted.shape
  E,_,F = w_in.shape

  blocks_per_expert = (group_sizes + BLOCK_M-1)//BLOCK_M

  max_blocks = int((M+E*(BLOCK_M-1))//BLOCK_M)

  padded_M = max_blocks * BLOCK_M


  block_expert_ids = jnp.repeat(jnp.arange(E, dtype=jnp.int32), blocks_per_expert, total_repeat_length=max_blocks)

  group_starts = jnp.concatenate([jnp.zeros(1, dtype=jnp.int32), jnp.cumsum(group_sizes)[:-1]])
  padded_group_starts = jnp.concatenate([jnp.zeros(1, dtype=jnp.int32), jnp.cumsum(blocks_per_expert * BLOCK_M)[:-1]])

  group_starts_rep = jnp.repeat(group_starts, group_sizes, total_repeat_length=M)
  padded_starts_rep = jnp.repeat(padded_group_starts, group_sizes, total_repeat_length=M)

  local_offsets = jnp.arange(M, dtype=jnp.int32) - group_starts_rep
  compact_offsets = padded_starts_rep + local_offsets

  x_padded = jnp.zeros((padded_M, D), dtype=x_sorted.dtype).at[compact_offsets].set(x_sorted)


  kernel = partial(fused_ragged_moe_kernel, BLOCK_M=BLOCK_M, BLOCK_F=BLOCK_F, D=D, F=F, MAX_BLOCKS=max_blocks)


  y_padded = pl.pallas_call(kernel, grid=(max_blocks,), out_shape=jax.ShapeDtypeStruct((padded_M, D), x_sorted.dtype))(x_padded, w_in, w_out, block_expert_ids)

  return y_padded[compact_offsets]

Training

In [ ]:
config = Config

model = LanguageModel(config)

rng = jax.random.PRNGKey(42)

mock_data = jax.ShapeDtypeStruct((1,MAX_SEQ_LEN-1), dtype=jnp.int32)

#lazy init is supposed to be faster than using model.apply

params = model.lazy_init(rng, mock_data)

tx = optax.adamw(learning_rate=3e-4, weight_decay = 1e-2)

state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)

def loss_fn(params, x, y):
  logits = model.apply(params,x)
  loss = optax.softmax_cross_entropy_with_integer_labels(logits,y)
  return loss.mean()


def train_step(state,x,y):
     loss, grads = jax.value_and_grad(loss_fn)(state.params,x,y)
     state = state.apply_gradients(grads=grads)
     return loss, state

#training loop
BATCH_SIZE=256
STEPS=5000

for i in range(STEPS):
  x,y = create_batches(BATCH_SIZE)
  loss, state = train_step(state,x,y)
  if i%100==0:
    print(f"{i}:{loss}")



Forward Pass Comparison (jax.lax.ragged_dot vs fused_kernel)

In [19]:
import time
#pallas(p) ragged(r)
p_config = Config(d_model=128, n_layers=5, n_heads=4, use_moe=True, n_experts=6, ff_d_ratio=32, use_kernel=True)
r_config = Config(d_model=128, n_layers=5, n_heads=4, use_moe=True, n_experts=6, ff_d_ratio=32, use_kernel=False)

p_model = LanguageModel(p_config)
r_model = LanguageModel(r_config)

rng = jax.random.PRNGKey(42)

mock_data = jax.ShapeDtypeStruct((1,MAX_SEQ_LEN-1), dtype=jnp.int32)

#lazy init is supposed to be faster than using model.apply

p_params = p_model.lazy_init(rng, mock_data)
r_params = r_model.lazy_init(rng, mock_data)

BATCH_SIZE=256


x,y = create_batches(BATCH_SIZE)


p_apply_jit = jax.jit(p_model.apply)
r_apply_jit = jax.jit(r_model.apply)

#compile
_ = p_apply_jit(p_params,x).block_until_ready()
_ = r_apply_jit(r_params,x).block_until_ready()

print("warmup done")


RUNS=10
#Timer
p_start_time = time.perf_counter()
for _ in range(RUNS):
  p_logits = p_apply_jit(p_params,x).block_until_ready()

p_end_time = time.perf_counter()

p_avg_time = (p_end_time - p_start_time)/RUNS

print(f"pallas: {p_avg_time:.4f} seconds")


#Timer
r_start_time = time.perf_counter()
for _ in range(RUNS):
  r_logits = r_apply_jit(r_params,x).block_until_ready()

r_end_time = time.perf_counter()

r_avg_time = (r_end_time - r_start_time)/RUNS

print(f"ragged: {r_avg_time:.4f} seconds")



warmup done
pallas: 0.0016 seconds
ragged: 0.0018 seconds
